In [8]:
# !nvidia smi
torch.cuda.get_device_name(0)

'Tesla T4'

**CUDA**: Nvidia's platform for writing code that runs on GPU core's in parallel.When PyTorch does matrix math, CUDA is doing the actual work on the chip. Apple's equivalent is called Metal/MPS.

**VRAM**: Dedicated memory on an Nvidia GPU chip, seperate from your system ram. If your model is bigger than VRAM, it crashes.

**fp16 (half precision)** — Instead of using 32 bits to store each number (fp32), use 16 bits. Half the memory, nearly same accuracy for inference. A 7B model in fp32 = 28GB. Same model in fp16 = 14GB.

**Tensor Cores**: Special hardware units NVIDIA GPU's built specifically to do matrix multiplication fast. Regular GPU cores can do it, but Tensor Cores are 4-8x faster.


In [4]:
import torch
import time

size = 5000
a_cpu = torch.randn(size, size)
b_cpu = torch.randn(size, size)

start = time.time()
c_cpu = a_cpu @ b_cpu
cpu_time = time.time() - start
print(f"CPU time: {cpu_time:.4f} seconds")

CPU time: 1.8935 seconds


In [5]:
if torch.cuda.is_available():
    a_gpu = a_cpu.to("cuda")
    b_gpu = b_cpu.to("cuda")

    torch.cuda.synchronize()
    start = time.time()
    c_gpu = a_gpu @ b_gpu
    torch.cuda.synchronize()
    gpu_time = time.time() - start
    print(f"GPU: {gpu_time:.3f}s")
    print(f"Speedup: {cpu_time / gpu_time:.0f}x")

GPU: 0.202s
Speedup: 9x


In [6]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"Allocated: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")
print(f"Free: {torch.cuda.memory_reserved(0) / 1e9:.2f} GB")

GPU: Tesla T4
Total VRAM: 15.6 GB
Allocated: 0.31 GB
Free: 0.32 GB


**2 bytes per parameter for fp16**

7B parameter model × 2 bytes = 14GB needed


13B model = 26GB needed

If you have 16GB unified memory → you can fit a 7B model, barely